# Tome C3, Chapter 1: Finite Automata (Python companion)

Exercises first, solutions at the end. We build a small automaton library and use it to check every construction of the chapter: the automata of multiples, the subset construction and its exponential price, Thompson's construction and a regular-expression engine that cannot be fooled by backtracking, the McNaughton–Yamada expressions, Moore's minimisation tested against the Myhill–Nerode theorem, one-letter automata and the Frobenius problem, and synchronising words with Černý's automata.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, itertools, time, re, math
from collections import deque
import matplotlib.pyplot as plt
random.seed(1)

## Exercises

**P1.1 (★) Multiples.** Implement DFAs and the automaton of Proposition 1.2. Check it on random numbers, and tabulate the size of its minimal automaton for bases $b\le 12$ and moduli $m\le12$ (Exercise 1.1).

**P1.2 (★) Guessing has a price.** Implement NFAs with $\varepsilon$-moves and the subset construction (Theorem 1.4), building only reachable subsets. Apply it to the NFA of $L_n$ (Proposition 1.5) and count the states.

**P1.3 (★★) A regex engine.** Parse regular expressions with `+`, concatenation, `*` and parentheses, compile them by Thompson's construction (Theorem 1.7), and match by simulating the set of states. Compare with Python's `re` on random words, then time both on $(a?)^ka^k$ against $a^k$.

**P1.4 (★★) From automata back to expressions.** Implement the McNaughton–Yamada construction and check that the expression it produces for the multiples of $3$ denotes the right language. How large are the expressions for the multiples of $m$?

**P1.5 (★★) Minimisation.** Implement Moore's algorithm (Theorem 1.15) and check it on random DFAs against a brute-force computation of the Myhill–Nerode classes (Theorem 1.14). Count the refinement rounds.

**P1.6 (★) One letter.** Compute the tail and cycle of random one-letter DFAs, and the minimal DFA of $(a^p+a^q)^*$ (Exercise 1.8).

**P1.7 (★★) Synchronising words.** Compute shortest synchronising words by breadth-first search on subsets. Check that Černý's automaton with $n$ states needs exactly $(n-1)^2$ letters, and look at random automata (Exercise 1.9).

## Solutions

### P1.1 Multiples

In [ ]:
class DFA:
    def __init__(self, states, alphabet, delta, start, accept):
        self.Q, self.S, self.d, self.q0, self.F = list(states), list(alphabet), dict(delta), start, set(accept)
    def run(self, w, q=None):
        q = self.q0 if q is None else q
        for a in w: q = self.d[q, a]
        return q
    def accepts(self, w): return self.run(w) in self.F

def multiples(b, m):
    return DFA(range(m), range(b), {(r, d): (b * r + d) % m for r in range(m) for d in range(b)}, 0, {0})

A = multiples(10, 7)
ok = all(A.accepts([int(c) for c in str(n)]) == (n % 7 == 0) for n in random.sample(range(10**12), 2000))
print("multiples of 7 in base 10, 2000 random numbers:", ok)
print("1001 in binary, states visited:", [multiples(2, 3).run([1, 0, 0, 1][:k]) for k in range(5)])

For the table we need minimisation, which is P1.5; we write Moore's algorithm now. Each round splits classes according to the classes of the successors.

In [ ]:
def reachable(A):
    seen, todo = {A.q0}, [A.q0]
    while todo:
        q = todo.pop()
        for a in A.S:
            r = A.d[q, a]
            if r not in seen: seen.add(r); todo.append(r)
    return seen

def moore(A):
    Q = [q for q in A.Q if q in reachable(A)]
    cls = {q: int(q in A.F) for q in Q}; rounds = 0
    while True:
        sig = {q: (cls[q],) + tuple(cls[A.d[q, a]] for a in A.S) for q in Q}
        ids = {s: i for i, s in enumerate(sorted(set(sig.values())))}
        new = {q: ids[sig[q]] for q in Q}
        if len(set(new.values())) == len(set(cls.values())):
            return cls, rounds
        cls, rounds = new, rounds + 1

def minimal_size(A): return len(set(moore(A)[0].values()))

print("     m:" + "".join(f"{m:4d}" for m in range(1, 13)))
for b in range(2, 13):
    print(f"b = {b:2d}:" + "".join(f"{minimal_size(multiples(b, m)):4d}" for m in range(1, 13)))

When $\gcd(b,m)=1$ the entry is $m$, as Exercise 1.1 proves. Otherwise it can be much smaller: in base $10$, multiples of $2$, $5$ and $10$ need only $2$ states (look at the last digit), multiples of $4$ need $3$ and multiples of $8$ need $5$, while multiples of $12$ need $7$. A multiple of $m=m_1m_2$, with $m_1$ prime to $b$ and every prime factor of $m_2$ dividing $b$, is a multiple of $m_1$ whose last few digits form a multiple of $m_2$; can you derive the entries of the table from this?

### P1.2 Guessing has a price

In [ ]:
class NFA:
    def __init__(self, delta, start, accept):   # delta: dict (q, a or '') -> set of states
        self.d, self.q0, self.F = delta, start, set(accept)
    def closure(self, S):
        S, todo = set(S), list(S)
        while todo:
            q = todo.pop()
            for r in self.d.get((q, ''), ()):
                if r not in S: S.add(r); todo.append(r)
        return frozenset(S)
    def step(self, S, a):
        return self.closure({r for q in S for r in self.d.get((q, a), ())})
    def accepts(self, w):
        S = self.closure({self.q0})
        for a in w: S = self.step(S, a)
        return bool(S & self.F)

def subset_construction(N, alphabet):
    s0 = N.closure({N.q0}); seen = {s0: 0}; todo = [s0]; delta = {}
    while todo:
        S = todo.pop()
        for a in alphabet:
            T = N.step(S, a)
            if T not in seen: seen[T] = len(seen); todo.append(T)
            delta[seen[S], a] = seen[T]
    return DFA(range(len(seen)), alphabet, delta, 0, {i for S, i in seen.items() if S & N.F})

def nfa_Ln(n):
    d = {(0, 'a'): {0, 1}, (0, 'b'): {0}}
    for i in range(1, n): d[i, 'a'] = d[i, 'b'] = {i + 1}
    return NFA(d, 0, {n})

for n in range(1, 13):
    D = subset_construction(nfa_Ln(n), 'ab')
    print(f"n = {n:2d}: NFA {n+1:2d} states, subset DFA {len(D.Q):5d} states, minimal {minimal_size(D):5d} = 2^{n}")
D = subset_construction(nfa_Ln(4), 'ab')
print("subset DFA agrees with the definition on all words of length <= 10:",
      all(D.accepts(w) == (len(w) >= 4 and w[-4] == 'a') for k in range(11) for w in itertools.product('ab', repeat=k)))

The reachable subsets are exactly $2^n$ and none can be merged: the subset construction is optimal here, and the NFA is exponentially more compact than any DFA, as Proposition 1.5 proves.

### P1.3 A regex engine

In [ ]:
def parse(s):
    # grammar: expr = term ('+' term)*, term = factor*, factor = atom '*'*, atom = letter | 'ε' | '∅' | '(' expr ')'
    pos = 0
    def peek(): return s[pos] if pos < len(s) else None
    def expr():
        nonlocal pos
        t = term()
        while peek() == '+':
            pos += 1; t = ('+', t, term())
        return t
    def term():
        fs = []
        while peek() is not None and peek() not in '+)':
            fs.append(factor())
        if not fs: return ('eps',)
        t = fs[0]
        for f in fs[1:]: t = ('.', t, f)
        return t
    def factor():
        nonlocal pos
        a = atom()
        while peek() == '*':
            pos += 1; a = ('*', a)
        return a
    def atom():
        nonlocal pos
        c = peek(); pos += 1
        if c == '(':
            e = expr(); assert peek() == ')'; pos += 1; return e
        if c == 'ε': return ('eps',)
        if c == '∅': return ('empty',)
        return ('sym', c)
    e = expr(); assert pos == len(s), "trailing input"
    return e

def thompson(e):
    d, count = {}, [0]
    def new():
        count[0] += 1; return count[0] - 1
    def add(p, a, q): d.setdefault((p, a), set()).add(q)
    def build(e):
        s, f = new(), new()
        if e[0] == 'sym': add(s, e[1], f)
        elif e[0] == 'eps': add(s, '', f)
        elif e[0] == '+':
            for sub in e[1:]:
                a, b = build(sub); add(s, '', a); add(b, '', f)
        elif e[0] == '.':
            a1, b1 = build(e[1]); a2, b2 = build(e[2])
            add(s, '', a1); add(b1, '', a2); add(b2, '', f)
        elif e[0] == '*':
            a, b = build(e[1]); add(s, '', a); add(s, '', f); add(b, '', a); add(b, '', f)
        return s, f
    s, f = build(e)
    return NFA(d, s, {f}), count[0]

def to_python(s): return s.replace('+', '|').replace('ε', '')

tests = ["(a+b)*a(a+b)(a+b)", "(0+1(01*0)*1)*", "(ab+ba)*", "a*b*a*", "((a+b)(a+b))*", "(a+ε)(ba)*b*"]
for t in tests:
    N, size = thompson(parse(t)); alph = sorted(set(t) & set('ab01'))
    words = [''.join(w) for k in range(9) for w in itertools.product(alph, repeat=k)]
    agree = all(N.accepts(w) == bool(re.fullmatch(to_python(t), w)) for w in words)
    print(f"{t:22s}: {size:3d} NFA states, agrees with re on {len(words)} words: {agree}")

In [ ]:
def time_it(f, *args):
    t = time.perf_counter(); f(*args); return time.perf_counter() - t

ks, t_re, t_nfa = list(range(4, 25, 2)), [], []
for k in ks:
    pat = "(a+ε)" * k + "a" * k
    pyre = re.compile("(?:a?)" * k + "a" * k)
    t_re.append(time_it(pyre.fullmatch, "a" * k))
    N, _ = thompson(parse(pat))
    t_nfa.append(time_it(N.accepts, "a" * k))
    print(f"k = {k:2d}: re {t_re[-1]:9.5f} s   automaton {t_nfa[-1]:8.5f} s")
plt.figure(figsize=(5, 3))
plt.semilogy(ks, t_re, "o-", label="Python re (backtracking)")
plt.semilogy(ks, t_nfa, "s-", label="set of states (Thompson)")
plt.xlabel("k"); plt.ylabel("seconds"); plt.legend(); plt.tight_layout(); plt.show()

Our engine agrees with Python's on every word tested. On $(a?)^ka^k$ against $a^k$ the backtracking engine tries the optional letters in all possible ways before finding the one that works, and its time doubles with each step of $k$; the simulation of the automaton keeps at most $O(k)$ states at each letter and stays polynomial. This is the example of Russ Cox's 2007 essay, and the reason why some libraries (RE2, Rust's `regex`) are built on automata.

### P1.4 From automata back to expressions

In [ ]:
def union(r, s):
    if r is None: return s
    if s is None: return r
    return r if r == s else f"{r}+{s}"
def concat(*parts):
    if any(p is None for p in parts): return None
    parts = [p for p in parts if p != 'ε']
    if not parts: return 'ε'
    return ''.join(p if ('+' not in p) else f"({p})" for p in parts)
def star(r):
    if r is None or r == 'ε': return 'ε'
    return f"{r}*" if len(r) == 1 else f"({r})*"

def mcnaughton_yamada(A):
    Q = list(A.Q); n = len(Q)
    R = {(i, j): None for i in Q for j in Q}            # None denotes the empty set
    for i in Q:
        for a in A.S:
            R[i, A.d[i, a]] = union(R[i, A.d[i, a]], str(a))
        R[i, i] = 'ε' if R[i, i] is None else f"ε+{R[i, i]}"
    for k in Q:
        R = {(i, j): union(R[i, j], concat(R[i, k], star(R[k, k]), R[k, j])) for i in Q for j in Q}
    out = None
    for f in A.F: out = union(out, R[A.q0, f])
    return out if out is not None else '∅'

r3 = mcnaughton_yamada(multiples(2, 3))
print("expression for multiples of 3:", r3[:200], "..." if len(r3) > 200 else "", f"({len(r3)} symbols)")
N, _ = thompson(parse(r3))
print("it denotes the multiples of 3 (all words up to length 12):",
      all(N.accepts(''.join(w)) == (int(''.join(w) or '0', 2) % 3 == 0) for k in range(13) for w in itertools.product('01', repeat=k)))
N2, _ = thompson(parse("(0+1(01*0)*1)*"))
print("Arden's expression of Exercise 1.4 agrees:", all(N2.accepts(''.join(w)) == (int(''.join(w) or '0', 2) % 3 == 0) for k in range(13) for w in itertools.product('01', repeat=k)))
for m in range(2, 9):
    print(f"multiples of {m} in binary: McNaughton-Yamada expression of {len(mcnaughton_yamada(multiples(2, m))):8d} symbols")

The construction is correct but verbose: even for $m=3$ it produces a long expression where Arden's lemma gives $(0+1(01^*0)^*1)^*$, and the length grows roughly by a constant factor with each new state, as the recurrence $R^k=R^{k-1}+R^{k-1}(R^{k-1})^*R^{k-1}$ suggests. Simplifying regular expressions is itself a hard problem (Chapter 10 will say how hard).

### P1.5 Minimisation

In [ ]:
def random_dfa(n, alphabet='ab', p=0.3):
    d = {(q, a): random.randrange(n) for q in range(n) for a in alphabet}
    return DFA(range(n), alphabet, d, 0, {q for q in range(n) if random.random() < p})

def brute_classes(A, depth):
    # Myhill-Nerode on the reachable states: signature = acceptance of every word of length <= depth
    words = [w for k in range(depth + 1) for w in itertools.product(A.S, repeat=k)]
    return {q: tuple(A.run(w, q) in A.F for w in words) for q in reachable(A)}

ok, rounds = True, []
for trial in range(300):
    n = random.randint(2, 9); A = random_dfa(n)
    cls, r = moore(A); rounds.append(r)
    sig = brute_classes(A, n)                            # words of length < n suffice
    same = lambda p, q: sig[p] == sig[q]
    ok &= all((cls[p] == cls[q]) == same(p, q) for p in cls for q in cls)
print("Moore's classes = brute-force indistinguishability on 300 random DFAs:", ok)
print("refinement rounds: max", max(rounds), " mean", round(sum(rounds) / len(rounds), 2))
reach, sizes = [], []
for _ in range(30):
    A = random_dfa(200); reach.append(len(reachable(A))); sizes.append(minimal_size(A))
print(f"random DFAs with 200 states over {{a,b}}: reachable {min(reach)} to {max(reach)}, minimal {min(sizes)} to {max(sizes)}, equal in {sum(r == m for r, m in zip(reach, sizes))} of 30")

Moore's algorithm agrees with the brute-force Myhill–Nerode classes every time, and needs very few rounds on random automata, far below the worst case $n-2$ of Exercise 1.7. On random automata with $200$ states, almost all the reduction comes from unreachable states: once those are removed, a random automaton is usually already minimal (26 of our 30 were). Minimisation earns its keep mostly on automata built by constructions, such as Thompson's and the subset construction, which are full of redundant states.

### P1.6 One letter

In [ ]:
def lasso(A):
    seen, q, i = {}, A.q0, 0
    while q not in seen:
        seen[q] = i; q = A.d[q, 'a']; i += 1
    return seen[q], i - seen[q]                     # tail mu, cycle lambda

n = 10**4; tails, cycles = [], []
for _ in range(300):
    A = random_dfa(n, 'a'); mu, lam = lasso(A); tails.append(mu); cycles.append(lam)
print(f"random one-letter DFAs with {n} states: mean tail {sum(tails)/300:.0f}, mean cycle {sum(cycles)/300:.0f}, sqrt(pi n/8) = {math.sqrt(math.pi*n/8):.0f}")

def semigroup_dfa(p, q):
    mu = (p - 1) * (q - 1)
    S = {i * p + j * q for i in range(mu + 1) for j in range(mu + 1)}
    return DFA(range(mu + 1), 'a', {(k, 'a'): min(k + 1, mu) for k in range(mu + 1)}, 0, {k for k in range(mu + 1) if k in S})
for p, q in [(2, 3), (3, 5), (4, 7), (5, 8), (6, 7)]:
    A = semigroup_dfa(p, q)
    N, _ = thompson(parse(f"({'a'*p}+{'a'*q})*"))
    agree = all(A.accepts('a' * k) == N.accepts('a' * k) for k in range(80))
    print(f"(a^{p}+a^{q})*: minimal DFA {minimal_size(A)} states = (p-1)(q-1)+1 = {(p-1)*(q-1)+1}; agrees with the expression: {agree}; largest gap {p*q-p-q}")

The one-letter automaton is a rho, like the iteration of Pollard's method in Tome C2: from a random transition function, the tail and the cycle both have mean about $\sqrt{\pi n/8}$, the birthday paradox again. For two coprime step lengths, the minimal automaton is a path of $(p-1)(q-1)+1$ states ending in an accepting loop, as Exercise 1.8 predicts, and the largest rejected length is Sylvester's $pq-p-q$.

### P1.7 Synchronising words

In [ ]:
def shortest_reset(A):
    start = frozenset(A.Q); dist = {start: 0}; todo = deque([start])
    while todo:
        S = todo.popleft()
        if len(S) == 1: return dist[S]
        for a in A.S:
            T = frozenset(A.d[q, a] for q in S)
            if T not in dist: dist[T] = dist[S] + 1; todo.append(T)
    return None                                      # no synchronising word

def cerny(n):   # a: rotate, b: merge state n-1 into 0
    d = {}
    for q in range(n):
        d[q, 'a'] = (q + 1) % n
        d[q, 'b'] = 0 if q == n - 1 else q
    return DFA(range(n), 'ab', d, 0, set())

for n in range(2, 11):
    print(f"Cerny automaton, n = {n:2d}: shortest synchronising word {shortest_reset(cerny(n)):3d} = (n-1)^2 = {(n-1)**2}")
lens = {}
for n in [5, 8, 11, 14]:
    vals = [shortest_reset(random_dfa(n)) for _ in range(200)]
    syn = [v for v in vals if v is not None]
    lens[n] = syn
    print(f"random DFAs, n = {n:2d}: {len(syn)}/200 synchronising, mean length {sum(syn)/len(syn):.2f}, max {max(syn)}, (n-1)^2 = {(n-1)**2}")

Černý's automata need exactly $(n-1)^2$ letters. Most random automata are synchronising (more than $90\%$ of them from $n=8$ on in our sample), with very short reset words whose length grows slowly with $n$. The conjecture that $(n-1)^2$ always suffices has been checked by computer for small $n$ and is still open in general; the search above is exponential in $n$, and that is no accident: finding a shortest synchronising word is NP-hard (Eppstein, 1990), a notion of Chapter 9.